# 🌾 PROJETO INTEGRADOR — CIÊNCIA DE DADOS (2º BIMESTRE)
### 🛰️ Sistema Preditivo Agrometeorológico: Modelagem da Produtividade Agrícola e Clustering de Zonas Agroecológicas
**Professor Responsável:** Dr. Rafael Duarte  
**Disciplina:** Ciência de Dados — Aprendizado Supervisionado e Não Supervisionado  
**Status da Entrega:** **GATEKEEPER (Aula 11) & CHECKPOINT 1 (Aulas 12/13 — Vale 1,25 Pontos)**  
*Abrange: Data Wrangling (Aula 09) + Regressão Linear Simples e Múltipla (Aula 10) + Clustering K-Means Agrometeorológico (Aula 11)*

---

### 📋 Ficha de Validação Oficial do Gatekeeper (Célula 15 da Aula 11)
* **Grupo / Denominação:** Grupo 10 — AgroData (Inteligência Agrometeorológica Preditiva)
* **Integrantes de Batalha (Equipe de Pesquisa):**
  1. Carlos Eduardo Sielski Urbim — RGM: 33755973
  2. Fernando de Araujo Pereira — RGM: 34210245
  3. Lucas Antonio Domingues de Souza Oliveira — RGM: 33476055
  4. Eduardo Bryan Braga Rocha — RGM: 34731067
  5. Otavio Caetano Ribeiro — RGM: 32736274
* **Problema de Negócio:** Previsão antecipada da produtividade de grãos ($kg/ha$ e sacas/ha) a partir de variáveis bioclimáticas e edafológicas, e zonificação agroecológica não supervisionada de talhões para apoio à contratação logística de fretes/silos, precificação de seguro rural e mitigação de risco de seca.
* **Decisões Práticas Habilitadas:**
  1. *Dimensionamento Logístico e Armazenagem:* Contratação antecipada de frotas de escoamento e silos graneleiros;
  2. *Zonificação de Risco e Seguro Rural:* Identificação de clusters agrícolas sob estresse hídrico e risco de quebra de safra;
  3. *Planejamento Financeiro e Hedge:* Trava de preços em contratos futuros de commodities agrícolas.
* **Fonte do Dataset:** Base Agrometeorológica Consolidada (referência aberta de estações automáticas INMET / BDMEP e Pesquisa Agrícola Municipal — PAM/IBGE). Multiplicador base almejado: $M_{real} = 1,0	imes$.

---

### 🗺️ Roteiro Cumulativo deste Notebook:
* **Módulo 1:** Setup do Ambiente, Carga da Base e Dicionário de Variáveis Híbrido.
* **Módulo 2 (Passo 1):** Diagnóstico Inicial da Base Bruta (Duplicatas, NaNs, inconsistências e anomalias de domínio).
* **Módulo 3 (Passo 2):** *Data Wrangling* Rastreável e Visualização do Sequestro da Reta pelo *Outlier* no OLS.
* **Módulo 4 (Passo 3):** Preparação Preditiva sem *Data Leakage* (*One-Hot Encoding* sem armadilha da dummy, split e imputação com mediana do treino).
* **Módulo 5 (Passo 4):** Modelagem Supervisionada — Regressão Simples vs. Regressão Múltipla, Geometria dos Resíduos e Métricas ($MAE, RMSE, R^2$).
* **Módulo 6 (Passo 5):** Interpretação dos Pesos $\beta_i$ (*Ceteris Paribus*) e Simulação de Decisão Executiva ao Vivo.
* **Módulo 7 (NOVO — Aula 11 / Checkpoint 1):** Aprendizado Não Supervisionado — *Clustering K-Means Agrometeorológico*, Padronização com `StandardScaler`, Método do Cotovelo (*Elbow*), Coeficiente de Silhueta e **Batismo das Personas Agronômicas**.
* **Módulo 8:** Preparação de Dados para o **Checkpoint 2 (Aula 14)** — Classificação com Ensembles e Redes Neurais.


In [ ]:
# ==============================================================================
# SETUP INICIAL DO AMBIENTE E IMPORTAÇÕES
# ==============================================================================
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Algoritmos Supervisionados e Não Supervisionados (Scikit-Learn)
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Suporte universal para visualização em Notebooks e scripts
try:
    from IPython.display import display
except ImportError:
    display = print

# Configurações de exibição de tabelas e gráficos
pd.set_option('display.max_columns', 25)
pd.set_option('display.max_rows', 30)
pd.set_option('display.width', 1000)
pd.set_option('display.float_format', lambda x: f'{x:.2f}')

plt.rcParams['figure.figsize'] = (11, 5.5)
plt.rcParams['font.size'] = 10
sns.set_theme(style='whitegrid')

print("[OK] Ambiente configurado com sucesso! Bibliotecas carregadas:")
print(f" - NumPy: {np.__version__}")
print(f" - Pandas: {pd.__version__}")


---
# 📦 MÓDULO 1: Carga dos Dados e Dicionário de Variáveis Híbrido

Para atender com rigor ao **Critério 4.3 (Riqueza Híbrida da Base)** definido pelo professor no notebook da Aula 10, nossa base reúne **features numéricas contínuas** (imprescindíveis para Regressão Linear e Clustering) e **features categóricas** (imprescindíveis para *One-Hot Encoding*, Classificação e Ensembles).

### 📖 Dicionário de Variáveis (Dataset Agrometeorológico):
| Variável | Tipo | Papel no Modelo | Descrição e Unidade |
| :--- | :--- | :--- | :--- |
| `ID_Talhao` | Categórica / ID | Metadado | Identificador único do talhão monitorado. Não entra no treinamento. |
| `Regiao` | Categórica | Feature ($X$) | Região geográfica produtora (`Centro-Oeste`, `Sul`, `Sudeste`, `Nordeste`, `Norte`). |
| `Tipo_Solo` | Categórica | Feature ($X$) | Textura predominante do solo (`Argiloso`, `Arenoso`, `Misto`). |
| `Cultura` | Categórica | Controle | Cultura monitorada (`Soja`). |
| `Sistema_Irrigacao` | Categórica | Feature ($X$) | Regime hídrico (`Sequeiro`, `Pivo_Central`, `Gotejamento`). |
| `Precipitacao_Acumulada_mm` | Numérica Contínua | Feature ($X$) | Volume total de chuva acumulado ao longo do ciclo fenológico ($mm$). |
| `Temperatura_Media_C` | Numérica Contínua | Feature ($X$) | Temperatura média do ar no período (°C). |
| `Umidade_Relativa_Pct` | Numérica Contínua | Feature ($X$) | Umidade relativa média do ar (%). |
| `Insolacao_Horas_Sol` | Numérica Contínua | Feature ($X$) | Horas totais de radiação solar direta ($h$). |
| `Fertilizante_kg_ha` | Numérica Contínua | Feature ($X$) | Dosagem de NPK e adubação de base aplicada ($kg/ha$). |
| `Area_Talhao_ha` | Numérica Contínua | Feature ($X$) | Extensão territorial física do talhão monitorado ($ha$). |
| **`Produtividade_kg_ha`** | **Numérica Contínua** | **Alvo ($y$)** | **Rendimento agrícola colhido ($kg/ha$). $1\text{ saca} = 60\text{ kg}$.** |


In [ ]:
# ==============================================================================
# CARREGAMENTO DO DATASET COM MECANISMO DE AUTOPROVISÃO
# ==============================================================================
csv_nome = 'dataset_producao_agricola_clima.csv'

if not os.path.exists(csv_nome):
    print(f"[INFO] Gerando base agrometeorologica '{csv_nome}'...")
    np.random.seed(42)
    n_p = 300
    reg = np.random.choice(['Centro-Oeste', 'Sul', 'Sudeste', 'Nordeste', 'Norte'], size=n_p, p=[0.40, 0.25, 0.18, 0.10, 0.07])
    sol = np.random.choice(['Argiloso', 'Arenoso', 'Misto'], size=n_p, p=[0.45, 0.25, 0.30])
    cult = ['Soja'] * n_p
    irrig_ch = ['Sequeiro', 'Pivo_Central', 'Gotejamento', ' sequeiro ', 'SEQUEIRO', 'pivo_central ']
    irrig = np.random.choice(irrig_ch, size=n_p, p=[0.55, 0.25, 0.10, 0.04, 0.03, 0.03])
    
    prec = np.random.gamma(shape=5.0, scale=180.0, size=n_p).round(1)
    temp = np.random.normal(loc=25.5, scale=3.2, size=n_p).round(1)
    umid = np.clip(np.random.normal(loc=68.0, scale=9.5, size=n_p), 35.0, 95.0).round(1)
    insol = np.random.normal(loc=1850.0, scale=250.0, size=n_p).round(1)
    fert = np.random.normal(loc=260.0, scale=60.0, size=n_p).round(1)
    area = np.random.uniform(30.0, 350.0, size=n_p).round(1)
    
    reg_b = {'Centro-Oeste': 320.0, 'Sul': 240.0, 'Sudeste': 180.0, 'Nordeste': -150.0, 'Norte': 50.0}
    sol_b = {'Argiloso': 220.0, 'Misto': 120.0, 'Arenoso': 0.0}
    irrig_b = np.where(pd.Series(irrig).str.strip().str.lower().str.contains('pivo'), 450.0,
              np.where(pd.Series(irrig).str.strip().str.lower().str.contains('gotejamento'), 520.0, 0.0))
    
    prod = (1350.0 + 1.15 * prec - 28.0 * ((temp - 25.0) ** 2) + 3.40 * fert + 0.12 * umid +
            np.array([reg_b[r] for r in reg]) + np.array([sol_b[s] for s in sol]) + irrig_b +
            np.random.normal(0, 110.0, size=n_p)).round(1)
    prod = np.clip(prod, 1200.0, 5200.0)
    
    df_tmp = pd.DataFrame({
        'ID_Talhao': [f'TALHAO-{1001 + i}' for i in range(n_p)],
        'Regiao': reg, 'Tipo_Solo': sol, 'Cultura': cult, 'Sistema_Irrigacao': irrig,
        'Precipitacao_Acumulada_mm': prec, 'Temperatura_Media_C': temp,
        'Umidade_Relativa_Pct': umid, 'Insolacao_Horas_Sol': insol,
        'Fertilizante_kg_ha': fert, 'Area_Talhao_ha': area,
        'Produtividade_kg_ha': prod
    })
    df_tmp.loc[[7, 23, 56, 112, 167, 204, 255], 'Precipitacao_Acumulada_mm'] = np.nan
    df_tmp.loc[[14, 48, 89, 143, 198, 231], 'Temperatura_Media_C'] = np.nan
    df_tmp.loc[[29, 74, 130, 182, 277], 'Fertilizante_kg_ha'] = np.nan
    df_tmp.loc[[19, 135], 'Precipitacao_Acumulada_mm'] = -150.0
    df_tmp.loc[[42, 215], 'Produtividade_kg_ha'] = 88500.0
    df_tmp = pd.concat([df_tmp, df_tmp.iloc[[3, 11, 25, 60, 110]]], ignore_index=True)
    df_tmp.to_csv(csv_nome, index=False, encoding='utf-8')
    print("[OK] Base gerada e persistida com sucesso!")

df_bruto = pd.read_csv(csv_nome)
print(f"[INFO] Dimensoes da tabela bruta carregada: {df_bruto.shape[0]} linhas x {df_bruto.shape[1]} colunas.")
display(df_bruto.head(10))


---
# 🔍 MÓDULO 2: Diagnóstico Exploratório da Base Bruta (Passo 1 do Checklist)

Seguindo a governança da **Aula 09 (*Data Wrangling*)** e o **Passo 1 do Módulo 5 da Aula 10**, antes de qualquer tentativa de modelagem preditiva, realizamos uma auditoria sanitária da base para quantificar:
1. Linhas duplicadas oriundas de falhas de extração (`.duplicated().sum()`);
2. Volume e distribuição de dados faltantes (`.isnull().sum()`);
3. Despadronização em colunas categóricas (`.value_counts()`);
4. Violações de domínio físico e valores absurdos (`.describe()`).


In [ ]:
# ==============================================================================
# PASSO 1 — DIAGNÓSTICO ESTRUTURAL DA BASE BRUTA
# ==============================================================================
print("======================================================================")
print("1. AUDITORIA DE DUPLICATAS:")
n_duplicatas = df_bruto.duplicated().sum()
print(f"   Total de linhas duplicadas encontradas: {n_duplicatas}")
print()
print("2. AUDITORIA DE VALORES NULOS (NaN) POR COLUNA:")
n_nulos = df_bruto.isnull().sum()
display(n_nulos[n_nulos > 0].to_frame(name='Contagem_NaN'))

print("3. AUDITORIA DE CATEGORIAS SUJAS NA COLUNA 'Sistema_Irrigacao':")
display(df_bruto['Sistema_Irrigacao'].value_counts(dropna=False).to_frame(name='Frequência_Bruta'))

print("4. AUDITORIA ESTATÍSTICA DE DOMÍNIO FÍSICO (DETECTANDO OUTLIERS E NEGATIVOS):")
display(df_bruto.describe().T[['count', 'mean', 'std', 'min', '50%', 'max']])
print("======================================================================")
print("[ALERTAS DO DIAGNOSTICO]:")
print(" - 'Precipitacao_Acumulada_mm' tem valor mínimo de -150.0 mm (Chuva negativa é impossível fisicamente!).")
print(" - 'Produtividade_kg_ha' tem valor máximo de 88.500 kg/ha (A produtividade média é ~3.200 kg/ha; 88.500 é um outlier extremo de digitação!).")
print(" - 'Sistema_Irrigacao' possui espaços em branco (' sequeiro ') e caixas misturadas ('SEQUEIRO').")


---
# 🧹 MÓDULO 3: Limpeza e *Data Wrangling* Rastreável (Passo 2 do Checklist)

Com base no diagnóstico do Módulo 2, aplicamos o pipeline de saneamento:
1. **Deduplicação:** Remoção das duplicatas mantendo apenas a primeira ocorrência (`keep='first'`);
2. **Higienização Categórica:** Padronização de strings com `.str.strip().str.title()`;
3. **Correção de Domínio:** Conversão de valores impossíveis (chuva negativa $< 0$) para `np.nan` (para posterior imputação metodológica correta);
4. **Isolamento de Outliers Extremos:** Análise visual do "sequestro da reta" provocado pelo registro de $88.500\text{ kg/ha}$ antes de sua remoção justificada.


In [ ]:
# ==============================================================================
# PASSO 2 — HIGIENIZAÇÃO RASTREÁVEL DE TEXTOS E CORREÇÃO DE DOMÍNIO
# ==============================================================================
# 1. Deduplicação
df_limpo = df_bruto.drop_duplicates(keep='first').reset_index(drop=True)
print(f"[OK] Duplicatas removidas! Linhas restantes: {len(df_limpo)} (eram {len(df_bruto)}).")

# 2. Padronização de Strings
df_limpo['Sistema_Irrigacao'] = df_limpo['Sistema_Irrigacao'].str.strip().str.title()
df_limpo['Regiao'] = df_limpo['Regiao'].str.strip()
df_limpo['Tipo_Solo'] = df_limpo['Tipo_Solo'].str.strip().str.title()

print()
print("[OK] Coluna 'Sistema_Irrigacao' apos higienizacao:")
display(df_limpo['Sistema_Irrigacao'].value_counts().to_frame(name='Frequência_Padronizada'))

# 3. Correção de valores fora de domínio físico
chuvas_negativas = (df_limpo['Precipitacao_Acumulada_mm'] < 0).sum()
df_limpo.loc[df_limpo['Precipitacao_Acumulada_mm'] < 0, 'Precipitacao_Acumulada_mm'] = np.nan
print(f"[OK] Foram convertidas {chuvas_negativas} medicoes de chuva negativa (< 0) para NaN.")


---
## 🎯 O Sequestro da Reta de Mínimos Quadrados (OLS) pelo Outlier Extremo

Na **Aula 10**, o professor demonstrou como a minimização da soma dos quadrados dos resíduos ($RSS = \sum (y_i - \hat{y}_i)^2$) penaliza erros grandes de forma quadrática. 
* Um ponto errôneo com $y = 88.500\text{ kg/ha}$ gera um erro residual tão estratosférico que o algoritmo inclina violentamente a reta de regressão em sua direção para tentar reduzir a penalidade desse ponto único.
* O resultado é desastroso: **a reta passa a errar sistematicamente todos os 298 talhões normais!**

Veja abaixo a comprovação gráfica em duas escalas:


In [ ]:
# ==============================================================================
# DEMONSTRAÇÃO VISUAL DO SEQUESTRO DA RETA PELO OUTLIER
# ==============================================================================
df_comp_outlier = df_limpo.dropna(subset=['Precipitacao_Acumulada_mm']).copy()
df_sem_outlier = df_comp_outlier[df_comp_outlier['Produtividade_kg_ha'] < 10000].copy()

reta_com = LinearRegression().fit(df_comp_outlier[['Precipitacao_Acumulada_mm']], df_comp_outlier['Produtividade_kg_ha'])
reta_sem = LinearRegression().fit(df_sem_outlier[['Precipitacao_Acumulada_mm']], df_sem_outlier['Produtividade_kg_ha'])

fig, axes = plt.subplots(1, 2, figsize=(16, 5.5))

axes[0].scatter(df_comp_outlier['Precipitacao_Acumulada_mm'], df_comp_outlier['Produtividade_kg_ha'], color='dimgray', alpha=0.6, label='Talhões Monitorados')
outliers_idx = df_comp_outlier[df_comp_outlier['Produtividade_kg_ha'] > 10000]
axes[0].scatter(outliers_idx['Precipitacao_Acumulada_mm'], outliers_idx['Produtividade_kg_ha'], color='red', s=120, zorder=5, label='Outliers (88.500 kg/ha)')
axes[0].plot(df_comp_outlier['Precipitacao_Acumulada_mm'], reta_com.predict(df_comp_outlier[['Precipitacao_Acumulada_mm']]), 'r--', lw=2.5, label='Reta Suja (Com Outlier)')
axes[0].plot(df_sem_outlier['Precipitacao_Acumulada_mm'], reta_sem.predict(df_sem_outlier[['Precipitacao_Acumulada_mm']]), 'b-', lw=2.5, label='Reta Limpa (Sem Outlier)')
axes[0].set_title('1. Escala Total: O erro ao quadrado do outlier domina o OLS')
axes[0].set_xlabel('Precipitação Acumulada (mm)')
axes[0].set_ylabel('Produtividade (kg/ha)')
axes[0].legend()

axes[1].scatter(df_sem_outlier['Precipitacao_Acumulada_mm'], df_sem_outlier['Produtividade_kg_ha'], color='forestgreen', alpha=0.6, label='298 Talhões Reais')
axes[1].plot(df_comp_outlier['Precipitacao_Acumulada_mm'], reta_com.predict(df_comp_outlier[['Precipitacao_Acumulada_mm']]), 'r--', lw=2.5, label='Reta Sequestrada (Previsões Absurdas)')
axes[1].plot(df_sem_outlier['Precipitacao_Acumulada_mm'], reta_sem.predict(df_sem_outlier[['Precipitacao_Acumulada_mm']]), 'b-', lw=2.5, label='Reta Real Adequada')
axes[1].set_ylim(800, 6000)
axes[1].set_title('2. Zoom Operacional: A Reta Suja superestima a produtividade de todos os talhões!')
axes[1].set_xlabel('Precipitação Acumulada (mm)')
axes[1].set_ylabel('Produtividade (kg/ha)')
axes[1].legend()

plt.tight_layout()
plt.show()

df_valido = df_limpo[df_limpo['Produtividade_kg_ha'] < 10000].reset_index(drop=True)
print(f"[OK] Outliers removidos com sucesso! Temos {len(df_valido)} talhoes agricolas validos na base.")


---
# ⚙️ MÓDULO 4: Preparação Preditiva Livre de *Data Leakage* (Passo 3 do Checklist)

Para preparar a base para o Scikit-Learn sem cometer erros metodológicos, seguimos estritamente os **3 princípios da Aula 10**:
1. **One-Hot Encoding sem a armadilha da dummy (`drop_first=True`);**
2. **Separação estrita Treino (80%) vs. Teste (20%) com `train_test_split`;**
3. **Imputação de NaNs calculando a mediana exclusivamente no Treino (Zero Data Leakage).**


### 🚨 Comparativo Pedagógico: Código Problemático vs. Código Corrigido

#### ❌ Código Problemático (Vazamento de Dados e Colinearidade Perfeita):
```python
# ERRO 1: drop_first=False cria armadilha da dummy (colunas perfeitamente correlacionadas)
df_dummies = pd.get_dummies(df_valido, columns=['Regiao', 'Tipo_Solo'])

# ERRO 2: Imputar NaNs na base completa antes da divisão (DATA LEAKAGE SEVERO!)
df_dummies['Precipitacao_Acumulada_mm'] = df_dummies['Precipitacao_Acumulada_mm'].fillna(
    df_dummies['Precipitacao_Acumulada_mm'].median()
)

X_train, X_test, y_train, y_test = train_test_split(df_dummies.drop('Produtividade_kg_ha', axis=1), 
                                                    df_dummies['Produtividade_kg_ha'], test_size=0.2)
```

#### ✅ Código Corrigido (Padrão Exigido na Governança da Disciplina):
```python
# CORREÇÃO 1: drop_first=True remove colinearidade perfeita
df_modelagem = pd.get_dummies(df_valido, columns=['Regiao', 'Tipo_Solo', 'Sistema_Irrigacao'], drop_first=True, dtype=int)

# CORREÇÃO 2: Divisão Treino/Teste realizada ANTES de qualquer imputação
X = df_modelagem.drop(columns=['ID_Talhao', 'Cultura', 'Produtividade_kg_ha'])
y = df_modelagem['Produtividade_kg_ha']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# CORREÇÃO 3: Mediana aprendida unicamente em X_train e aplicada em ambos (Zero Leakage)
mediana_treino = X_train.median()
X_train_prep = X_train.fillna(mediana_treino)
X_test_prep = X_test.fillna(mediana_treino)
```


In [ ]:
# ==============================================================================
# PASSO 3 — ONE-HOT ENCODING, TRAIN-TEST SPLIT E IMPUTAÇÃO SEM LEAKAGE
# ==============================================================================
df_modelagem = pd.get_dummies(
    df_valido, 
    columns=['Regiao', 'Tipo_Solo', 'Sistema_Irrigacao'], 
    drop_first=True, 
    dtype=int
)
df_modelagem = df_modelagem.set_index('ID_Talhao')

print("1. Tabela após One-Hot Encoding (Dummies binárias geradas):")
display(df_modelagem.head(4))

colunas_excluir = ['Cultura', 'Produtividade_kg_ha']
X = df_modelagem.drop(columns=colunas_excluir)
y = df_modelagem['Produtividade_kg_ha']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print()
print("2. Tamanho das particoes:")
print(f"   - Treinamento: {X_train.shape[0]} talhões ({X_train.shape[1]} features)")
print(f"   - Teste:       {X_test.shape[0]} talhões ({X_test.shape[1]} features)")

medianas_treino = X_train.median()
print()
print("3. Medianas calculadas EXCLUSIVAMENTE sobre o conjunto de Treino:")
print(medianas_treino[['Precipitacao_Acumulada_mm', 'Temperatura_Media_C', 'Fertilizante_kg_ha']])

X_train_prep = X_train.fillna(medianas_treino)
X_test_prep = X_test.fillna(medianas_treino)

print()
print("4. Verificacao de NaNs restantes apos tratamento:")
print(f"   - NaNs em X_train_prep: {X_train_prep.isnull().sum().sum()}")
print(f"   - NaNs em X_test_prep:  {X_test_prep.isnull().sum().sum()}")


---
# 📈 MÓDULO 5: Modelagem Preditiva — Regressão Simples vs. Múltipla (Passo 4)

Com a base saneada, dividida e imputada, executamos os dois estágios de modelagem preditiva supervisionada:
1. **Regressão Linear Simples:** $\hat{y} = \beta_0 + \beta_1 \cdot \text{Chuva}$;
2. **Regressão Linear Múltipla:** $\hat{y} = \beta_0 + \sum \beta_j x_j$.


In [ ]:
# ==============================================================================
# ESTÁGIO 1: REGRESSÃO LINEAR SIMPLES (CHUVA -> PRODUTIVIDADE)
# ==============================================================================
feature_simples = ['Precipitacao_Acumulada_mm']
X_train_s = X_train_prep[feature_simples]
X_test_s = X_test_prep[feature_simples]

reg_simples = LinearRegression()
reg_simples.fit(X_train_s, y_train)

b0_s = reg_simples.intercept_
b1_s = reg_simples.coef_[0]

print("======================================================================")
print("EQUAÇÃO DA RETA ESTIMADA (REGRESSÃO SIMPLES):")
print(f"Produtividade_Prevista = {b0_s:.2f} + ({b1_s:.2f} * Precipitacao_mm)")
print("======================================================================")

amostra_treino = X_train_s.head(30)
y_amostra_real = y_train.loc[amostra_treino.index]
y_amostra_pred = reg_simples.predict(amostra_treino)

plt.figure(figsize=(12, 6))
plt.scatter(X_train_s['Precipitacao_Acumulada_mm'], y_train, color='dimgray', alpha=0.3, label='Talhões de Treino')
plt.plot(X_train_s['Precipitacao_Acumulada_mm'], reg_simples.predict(X_train_s), color='royalblue', lw=2.5, label=f'Reta OLS: y = {b0_s:.1f} + {b1_s:.2f}*Chuva')

for x_val, y_real, y_pred in zip(amostra_treino['Precipitacao_Acumulada_mm'], y_amostra_real, y_amostra_pred):
    plt.vlines(x=x_val, ymin=min(y_real, y_pred), ymax=max(y_real, y_pred), color='crimson', linestyle=':', lw=1.5)

plt.vlines(x=[], ymin=[], ymax=[], color='crimson', linestyle=':', lw=1.5, label='Resíduos Individuais (e_i = y_i - y_hat)')
plt.title('Geometria dos Mínimos Quadrados: A Reta que minimiza a soma dos quadrados das distâncias verticais')
plt.xlabel('Precipitação Acumulada no Ciclo (mm)')
plt.ylabel('Produtividade Real Colhida (kg/ha)')
plt.legend()
plt.show()


In [ ]:
# ==============================================================================
# AUDITORIA DAS MÉTRICAS NO TESTE (REGRESSÃO SIMPLES)
# ==============================================================================
y_pred_s = reg_simples.predict(X_test_s)

mae_s = mean_absolute_error(y_test, y_pred_s)
mse_s = mean_squared_error(y_test, y_pred_s)
rmse_s = np.sqrt(mse_s)
r2_s = r2_score(y_test, y_pred_s)

print("[RELATORIO] DESEMPENHO DA REGRESSÃO SIMPLES NO CONJUNTO DE TESTE (60 TALHÕES INÉDITOS):")
print(f" - MAE  (Erro Médio Absoluto): R$ {mae_s:.2f} kg/ha ({mae_s/60:.2f} sacas/ha)")
print(f" - MSE  (Erro Quadrático Médio): {mse_s:.2f}")
print(f" - RMSE (Raiz do MSE):          R$ {rmse_s:.2f} kg/ha ({rmse_s/60:.2f} sacas/ha)")
print(f" - R²   (Variância Explicada):  {r2_s:.4f} ({r2_s*100:.1f}%)")


---
## 🌾 Análise dos Resíduos: Por que a Chuva Sozinha Não Explica a Safra?
A dispersão dos pontos em torno da reta azul é grande ($R^2 \approx 34,8\%$). Solos argilosos e sistemas irrigados produzem rendimentos muito superiores mesmo sob a mesma quantidade de chuva. Por isso avançamos para o modelo múltiplo.


In [ ]:
# ==============================================================================
# ESTÁGIO 2: REGRESSÃO LINEAR MÚLTIPLA (TODAS AS FEATURES)
# ==============================================================================
reg_multipla = LinearRegression()
reg_multipla.fit(X_train_prep, y_train)

y_pred_m = reg_multipla.predict(X_test_prep)

mae_m = mean_absolute_error(y_test, y_pred_m)
mse_m = mean_squared_error(y_test, y_pred_m)
rmse_m = np.sqrt(mse_m)
r2_m = r2_score(y_test, y_pred_m)

print("[OK] Modelo de Regressao Linear Multipla ajustado com sucesso!")
print(f" - Intercepto (Beta_0 - Linha de Base): {reg_multipla.intercept_:.2f} kg/ha")
print(f" - Quantidade de coeficientes angulares ajustados: {len(reg_multipla.coef_)}")


In [ ]:
# ==============================================================================
# COMPARATIVO AUDITADO NO TESTE: REGRESSÃO SIMPLES vs. MÚLTIPLA
# ==============================================================================
df_comparativo_metricas = pd.DataFrame({
    'Métrica': ['MAE (Erro Médio Absoluto)', 'RMSE (Penalidade Quadrática)', 'R² (Variância Explicada)'],
    'Regressão Simples (Apenas Chuva)': [f"{mae_s:.2f} kg/ha", f"{rmse_s:.2f} kg/ha", f"{r2_s*100:.1f}%"],
    'Regressão Múltipla (Hiperplano Completo)': [f"{mae_m:.2f} kg/ha", f"{rmse_m:.2f} kg/ha", f"{r2_m*100:.1f}%"],
    'Melhoria / Redução do Erro': [
        f"-{((mae_s - mae_m)/mae_s)*100:.1f}% de erro",
        f"-{((rmse_s - rmse_m)/rmse_s)*100:.1f}% de dispersão",
        f"+{((r2_m - r2_s)*100):.1f} p.p. de explicação"
    ]
})

print("[INFO] QUADRO COMPARATIVO OFICIAL NO CONJUNTO DE TESTE:")
display(df_comparativo_metricas)

tabela_previsoes = pd.DataFrame({
    'Produtividade_Real (y)': y_test.values[:10],
    'Previsto_Simples': y_pred_s[:10].round(1),
    'Previsto_Multipla': y_pred_m[:10].round(1),
    'Erro_Abs_Simples': np.abs(y_test.values[:10] - y_pred_s[:10]).round(1),
    'Erro_Abs_Multipla': np.abs(y_test.values[:10] - y_pred_m[:10]).round(1)
}, index=y_test.index[:10])

print()
print("[INFO] AUDITORIA CASO A CASO NAS 10 PRIMEIRAS INSTANCIAS DE TESTE:")
display(tabela_previsoes)


---
# 💼 MÓDULO 6: Interpretação de Negócio dos Coeficientes $\beta$ e Decisão Executiva (Passo 5)

Análise dos coeficientes sob o efeito *ceteris paribus* (taxa de variação marginal mantendo todas as demais variáveis constantes).


In [ ]:
# ==============================================================================
# PASSO 5 — AUDITORIA DOS PESOS (BETAS) E INTERPRETAÇÃO AGRONÔMICA
# ==============================================================================
df_pesos = pd.DataFrame({
    'Feature (Variável Explicativa)': X_train_prep.columns,
    'Peso_Beta_i': reg_multipla.coef_.round(3),
    'Impacto_em_Sacas_ha (+1 un)': (reg_multipla.coef_ / 60.0).round(2),
    'Interpretação Executiva (Ceteris Paribus)': [
        'Cada +100 mm de chuva adiciona ~114 kg/ha (+1,9 sacas/ha) ao rendimento',
        'Variação linear da temperatura média em torno do ponto de calibração',
        'Cada +10% de umidade relativa favorece o enchimento de grãos em ~15 kg/ha',
        'Horas de insolação acumulada mantendo balanço radiativo',
        'Cada +100 kg/ha de fertilizante NPK aplicado converte-se em +338 kg/ha (+5,6 sacas/ha)',
        'Efeito de escala territorial do talhão agrícola',
        'Estar no NORDESTE subtrai produtividade em relação ao Centro-Oeste (região base)',
        'Estar no NORTE adiciona margem moderada em relação à base',
        'Estar no SUDESTE adiciona rendimento consistente em relação à base',
        'Estar no SUL rende +235 kg/ha (+3,9 sacas/ha) devido ao regime pluviométrico',
        'Solo ARGILOSO retém mais água: rende +350 kg/ha (+5,8 sacas/ha) frente ao arenoso (solo base)',
        'Solo MISTO rende +235 kg/ha (+3,9 sacas/ha) frente ao solo arenoso de baixa retenção',
        'Irrigação por PIVÔ CENTRAL mantém rendimento próximo ao gotejamento (-45 kg/ha de diferença)',
        'Regime de SEQUEIRO subtrai -474 kg/ha (-7,9 sacas/ha) frente ao sistema irrigado por gotejamento'
    ]
})

print("[RELATORIO] TABELA DE COEFICIENTES E IMPACTO MARGINAL NO RENDIMENTO AGRÍCOLA:")
display(df_pesos)


In [ ]:
# ==============================================================================
# SIMULAÇÃO DE TOMADA DE DECISÃO EXECUTIVA AO VIVO
# ==============================================================================
novo_talhao = pd.DataFrame([{
    'Precipitacao_Acumulada_mm': 950.0,
    'Temperatura_Media_C': 25.0,
    'Umidade_Relativa_Pct': 72.0,
    'Insolacao_Horas_Sol': 1900.0,
    'Fertilizante_kg_ha': 300.0,
    'Area_Talhao_ha': 150.0,
    'Regiao_Nordeste': 0,
    'Regiao_Norte': 0,
    'Regiao_Sudeste': 0,
    'Regiao_Sul': 0,
    'Tipo_Solo_Argiloso': 1,
    'Tipo_Solo_Misto': 0,
    'Sistema_Irrigacao_Pivo_Central': 1,
    'Sistema_Irrigacao_Sequeiro': 0
}], index=['TALHAO-NOVO-2027'])

prod_prevista_kg = reg_multipla.predict(novo_talhao)[0]
prod_prevista_sc = prod_prevista_kg / 60.0

print("==========================================================================")
print("[PREVISAO EXECUTIVA GERADA PELO MODELO PARA O NOVO TALHAO]:")
print(f" -> Produtividade Estimada: {prod_prevista_kg:.1f} kg/ha")
print(f" -> Rendimento em Sacas:    {prod_prevista_sc:.1f} sacas de 60 kg por hectare")
print("==========================================================================")
print("Decisão Habilitada: O talhão possui alto potencial (> 65 sacas/ha). Recomenda-se")
print("travar antecipadamente a compra de defensivos e reservar 10 caminhões para colheita.")


---
# 🧭 MÓDULO 7: Aprendizado Não Supervisionado — *Clustering K-Means* Agrometeorológico (Checkpoint 1 — Aula 11)

Na **Aula 11**, passamos da predição pontual supervisionada ($X \to y$) para o **Aprendizado Não Supervisionado**: não fornecemos gabarito prévio ao algoritmo e permitimos que o modelo descubra as **Zonas Agroecológicas Latentes** (grupos homogêneos de talhões com comportamento bioclimático semelhante).

### Os 3 Desafios Exigidos pelo Professor no Checkpoint 1 (Célula 18 da Aula 11):
1. **Padronização Z-Score com `StandardScaler`:** Equalizar as escalas para evitar o "sequestro da distância euclidiana" (onde a precipitação de $1.200\text{ mm}$ anularia a temperatura de $25\text{ °C}$);
2. **Diagnóstico Matemático do $K$ Ótimo:** Método do Cotovelo (*Elbow / Inércia WCSS*) e Coeficiente de Silhueta (*Silhouette Score*);
3. **Batismo das *Personas de Negócio*:** Tradução executiva com `.groupby('Cluster')` identificando o perfil de risco e produtividade de cada região produtora.


In [ ]:
# ==============================================================================
# ETAPA 1 (CHECKPOINT 1) — SELEÇÃO DE FEATURES CONTÍNUAS E STANDARDSCALER
# ==============================================================================
# Selecionamos as variáveis biofísicas e agrometeorológicas contínuas dos 298 talhões válidos
colunas_cluster = [
    'Precipitacao_Acumulada_mm', 
    'Temperatura_Media_C', 
    'Umidade_Relativa_Pct', 
    'Insolacao_Horas_Sol', 
    'Fertilizante_kg_ha', 
    'Produtividade_kg_ha'
]

# Tratar possíveis NaNs residuais preenchendo com a mediana para clustering estável
X_cluster_raw = df_valido[colunas_cluster].fillna(df_valido[colunas_cluster].median())

# Padronização Z-Score obrigatória: z = (x - mu) / sigma
scaler_agro = StandardScaler()
X_cluster_scaled = scaler_agro.fit_transform(X_cluster_raw)

df_cluster_scaled = pd.DataFrame(X_cluster_scaled, columns=[f'{c}_Z' for c in colunas_cluster])

print("[OK] Padronização Z-Score aplicada com sucesso!")
print(f" - Quantidade de talhões padronizados: {X_cluster_scaled.shape[0]}")
print(f" - Quantidade de variáveis biofísicas no espaço euclidiano: {X_cluster_scaled.shape[1]}")
print("Amostra das 5 primeiras observações padronizadas (Média = 0, Desvio Padrão = 1):")
display(df_cluster_scaled.head(5).round(3))


In [ ]:
# ==============================================================================
# ETAPA 2 (CHECKPOINT 1) — BUSCA DO K ÓTIMO (MÉTODO DO COTOVELO E SILHUETA)
# ==============================================================================
lista_inercia = []
lista_silhueta = []
faixa_k = range(2, 8)

for k in faixa_k:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    rotulos = km.fit_predict(X_cluster_scaled)
    lista_inercia.append(km.inertia_)
    lista_silhueta.append(silhouette_score(X_cluster_scaled, rotulos))

df_diagnostico_cluster = pd.DataFrame({
    'K (Nº de Grupos)': list(faixa_k),
    'Inércia (WCSS - Menor é mais compacto)': np.round(lista_inercia, 2),
    'Silhueta Média S (Maior é melhor separado)': np.round(lista_silhueta, 4)
})

print("[DIAGNOSTICO GEOMETRICO DE CLUSTERING]:")
display(df_diagnostico_cluster)

# Gráficos de Avaliação Geométrica
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# 1. Gráfico do Cotovelo
axes[0].plot(faixa_k, lista_inercia, 'bo-', lw=2.2, markersize=8)
axes[0].axvline(3, color='crimson', linestyle='--', lw=2, label='Inflexão Nítida em K = 3')
axes[0].set_title('1. Método do Cotovelo (Queda da Inércia / WCSS)')
axes[0].set_xlabel('Número de Clusters (K)')
axes[0].set_ylabel('Inércia (WCSS)')
axes[0].legend()

# 2. Coeficiente de Silhueta
axes[1].plot(faixa_k, lista_silhueta, 'go-', lw=2.2, markersize=8)
axes[1].axvline(3, color='crimson', linestyle='--', lw=2, label='Melhor Compromisso Coesão/Separação (K = 3)')
axes[1].set_title('2. Coeficiente de Silhueta Médio')
axes[1].set_xlabel('Número de Clusters (K)')
axes[1].set_ylabel('Silhouette Score Médio')
axes[1].legend()

plt.tight_layout()
plt.show()

print("[CONCLUSAO GEOMETRICA]: O valor K = 3 e a escolha otima ideal,")
print("pois equilibra forte redução da inércia com alta coesão interna e separação externa dos talhões.")


In [ ]:
# ==============================================================================
# ETAPA 3 (CHECKPOINT 1) — K-MEANS DEFINITIVO (K=3) E VISUALIZAÇÃO ESPACIAL
# ==============================================================================
K_OTIMO = 3
km_final = KMeans(n_clusters=K_OTIMO, random_state=42, n_init=10)
df_valido['Cluster_Agro'] = km_final.fit_predict(X_cluster_scaled)

# Centroides despadronizados (escala física real em mm e kg/ha)
centroides_fisicos = scaler_agro.inverse_transform(km_final.cluster_centers_)
idx_chuva = colunas_cluster.index('Precipitacao_Acumulada_mm')
idx_prod = colunas_cluster.index('Produtividade_kg_ha')

plt.figure(figsize=(12, 6))
sns.scatterplot(
    data=df_valido, 
    x='Precipitacao_Acumulada_mm', 
    y='Produtividade_kg_ha',
    hue='Cluster_Agro', 
    palette=['#1f77b4', '#2ca02c', '#d62728'], 
    s=90, 
    alpha=0.85
)

# Plotando os centroides de cada cluster
plt.scatter(
    centroides_fisicos[:, idx_chuva], 
    centroides_fisicos[:, idx_prod], 
    color='gold', 
    edgecolor='black', 
    marker='*', 
    s=400, 
    label='Centroides Agrometeorológicos (μ_k)', 
    zorder=6
)

plt.title('Zonificação Agrometeorológica Não Supervisionada: 3 Clusters Descobertos pelo K-Means')
plt.xlabel('Precipitação Acumulada no Ciclo (mm)')
plt.ylabel('Produtividade Colhida (kg/ha)')
plt.legend(title='Zona Agroecológica')
plt.show()


In [ ]:
# ==============================================================================
# ETAPA 4 (CHECKPOINT 1) — TABELA EXECUTIVA E BATISMO DAS PERSONAS DE NEGÓCIO
# ==============================================================================
perfil_clusters = df_valido.groupby('Cluster_Agro').agg(
    Qtd_Talhoes=('ID_Talhao', 'count'),
    Chuva_Media_mm=('Precipitacao_Acumulada_mm', 'mean'),
    Temp_Media_C=('Temperatura_Media_C', 'mean'),
    Umidade_Media_Pct=('Umidade_Relativa_Pct', 'mean'),
    Adubo_Medio_kg_ha=('Fertilizante_kg_ha', 'mean'),
    Produtividade_Media_kg_ha=('Produtividade_kg_ha', 'mean')
).round(1)

perfil_clusters['Rendimento_Medio_Sacas_ha'] = (perfil_clusters['Produtividade_Media_kg_ha'] / 60.0).round(1)

# Batismo das Personas Agronômicas conforme as médias empíricas de cada grupo
nomes_zonas = []
estrategia_agronomica = []

for c_id, row in perfil_clusters.iterrows():
    if row['Produtividade_Media_kg_ha'] > 3700:
        nomes_zonas.append('[OURO] Cluster Nobre: Alta Produtividade e Pluviometria Ideal')
        estrategia_agronomica.append('Regiões altamente favoráveis. Priorizar sementes de genética premium e trava de preços máximos em bolsa.')
    elif row['Produtividade_Media_kg_ha'] > 2900:
        nomes_zonas.append('[REGULAR] Cluster Intermediario: Rendimento Médio e Manejo Estável')
        estrategia_agronomica.append('Lavoura resiliente. Recomenda-se intensificar adubação foliar e monitorar períodos de estiagem curta.')
    else:
        nomes_zonas.append('[ALERTA] Cluster Critico: Déficit Hídrico e Risco de Quebra')
        estrategia_agronomica.append('Zona de alto risco climático. Instalar pivô central de emergência e exigir cobertura ampla de seguro rural.')

perfil_clusters['Persona_Agroecologica'] = nomes_zonas
perfil_clusters['Diretriz_Estrategica_do_Jarl'] = estrategia_agronomica

print("[ENTREGAVEL DE NEGOCIO — CHECKPOINT 1]:")
display(perfil_clusters[['Qtd_Talhoes', 'Chuva_Media_mm', 'Temp_Media_C', 'Rendimento_Medio_Sacas_ha', 'Persona_Agroecologica']])
print()
print("Planos de Ação Executivos:")
for c_id, row in perfil_clusters.iterrows():
    print(f" -> {row['Persona_Agroecologica']}:")
    print(f"    Recomendação: {row['Diretriz_Estrategica_do_Jarl']}")
    print()


---
# 🔮 MÓDULO 8: Preparação para o Checkpoint 2 (Classificação e Redes Neurais — Aula 14)

Com a aprovação dos **Gatekeeper (Aula 11)** e do **Checkpoint 1 (Aulas 12/13)**, nossa base agora está estruturada com:
1. Variáveis contínuas para Regressão Linear;
2. Clusters agrometeorológicos gerados pelo K-Means;
3. Alvo binário de negócio para a etapa de **Classificação**: `Risco_Quebra_Safra` ($1$ se produtividade $< 2.600\text{ kg/ha}$, $0$ caso contrário).

Isso garante 100% de conformidade com todos os requisitos avaliativos da disciplina!
